# Offline inference
Internet off. Loads the packaged checkpoint and scores every study in the mounted test metadata.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import subprocess

asset = None
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    candidates = list(kaggle_input.iterdir()) + list(kaggle_input.glob('datasets/*/*'))
    for child in candidates:
        boot = child / 'bootstrap.py'
        if boot.is_file() and (child / 'assets' / 'checkpoint.pt').is_file():
            spec = importlib.util.spec_from_file_location('rsna_asset_bootstrap', boot)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            asset = module.mount_asset(child)
            break
if asset is None:
    raise SystemExit('BLOCKED: versioned code/weights asset is not mounted')
codec_target = Path('/tmp/rsna-offline-codecs')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--find-links', str(asset['root'] / 'wheels'), '--target', str(codec_target), '-r', str(asset['root'] / 'requirements.txt')], check=True)
sys.path.insert(0, str(codec_target))
sys.path.insert(0, str(asset['src']))
from rsna_knee.data.metadata import discover_input_root
from rsna_knee.submission.offline import run_offline_inference

data = discover_input_root()
if data is None:
    raise SystemExit('BLOCKED: competition input is not mounted')
result = run_offline_inference(
    checkpoint=Path(asset['checkpoint']),
    series_csv=data / 'test_series.csv',
    dicom_root=data / 'test_series',
    study_csv=data / 'test.csv',
    dest=Path('/kaggle/working/submission.csv'),
    work=Path('/kaggle/working/infer_cache'),
)
import json
from rsna_knee.hashing import sha256_file
Path('/kaggle/working/inference_report.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))


In [ ]:
import os
scoring_rerun = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in {'1', 'true'}
if not scoring_rerun and (data / 'train_series.csv').is_file() and (data / 'train_series').is_dir():
    # Decode a separate training subset to measure runtime and test preprocessing parity.
    import csv, importlib.metadata, resource
    from rsna_knee.data.preprocess import preprocess_hash
    benchmark_uids = ['1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260', '1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585', '1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231', '1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764', '1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542', '1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339', '1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101', '1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606', '1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513', '1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559', '1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394', '1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938', '1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806', '1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981', '1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018', '1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866', '1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801', '1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401', '1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269', '1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922', '1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736', '1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097', '1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018', '1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941', '1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347', '1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365', '1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449', '1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097', '1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678', '1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149', '1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863', '1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557', '1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364', '1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659', '1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636', '1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100', '1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882', '1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339', '1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343', '1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834', '1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833', '1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586', '1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166', '1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864', '1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471', '1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104', '1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182', '1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314', '1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824', '1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586', '1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725', '1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221', '1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019', '1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790', '1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205', '1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320', '1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930', '1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484', '1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536', '1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161', '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269', '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052', '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580', '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238', '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935', '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840', '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756', '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604', '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581', '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945', '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819', '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973', '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492', '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803', '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891', '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903', '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309', '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192', '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711', '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378', '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404', '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645', '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465', '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924', '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608', '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550', '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911', '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945', '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742', '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683', '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500', '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125', '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187', '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274', '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132', '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450', '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363', '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710', '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395', '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236', '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910', '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039', '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327', '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774', '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002', '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921', '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220', '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995', '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190', '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003', '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828', '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254', '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037', '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435', '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767', '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312', '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662']
    expected_shard_hashes = {'1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260': 'db974daf0b34148105422c2e21cba75324371391fdfd1bfbbef985522b8a2814', '1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585': '104921f6915c061fbc5fef0ee16511f7f8f86ef690fb51848a4f30244ad4e326', '1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231': '55ba7fd6fd22f50607bd910b0983207847b46f4342b3fb3e920bb93869358e3b', '1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764': '9492e53658a2662301fbf7f5cc67b303732710cbdd13a5b0633e8f3a8a6fc77b', '1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542': 'a6512aa03794ad90af91a49ce2789c64b2dd710ace0193e0568190db16b3bc60', '1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339': '1bb52170f82b97fb062305b94d7fdff66310d1524ffbbf963c2f09cc8e0e3682', '1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101': '2532e174dd523703224db8e74859126ed48ea186dff9f5ebaa75066707ea9712', '1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606': '8c3ad866c1cc8e5a4a0bcf395d50886f87661e8c3ed3761049ba8222fe325207', '1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513': '36d0d5a646f34ff2041ff715530fa8172f52b844fc14092d4048904bdbb4502f', '1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559': '211f6b56ad40bad55be3782dbbd1fc591ece5cb446fb52badc3f5e0539ae0a3c', '1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394': '9827c03ca12137b347fb3aa56adf6f001a88b05f223a15392cd6b161e6c2e469', '1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938': '56bbae696278e837055b573cfb6a88629d17a5b052b8dcc580198a90f7da01c0', '1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806': 'dfdc75ae512c4555f2627f79a51b6cf8b3ebb5de08ccb2e270986b75acce8e17', '1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981': 'f9b09772a0cce6a5a1228d19887d1b86b2d154e6be09271af1c7cb10b7e99094', '1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018': '98ef4cdb9a2751dc4563b30724eebe9ed67b1c19245f7e0ed50d7ce0b0aca30e', '1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866': 'e5a6ac55e1232547ff15756ba7519eb6c75699dd9c687fa4f6b2ed638dec9b39', '1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801': '21249fb6ebb2298c6483d4fd02cf8f848c0478c37772c7786b5eee5d5834aa37', '1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401': 'dedb1135029733e63dcbc1835bbf86b1555ec3bfc0aa8f13b16653b1df7cc4d3', '1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269': '9da13f5646c0e5a9bf000174aa80911669597664f9da83b0e4e3ea9b235dd423', '1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922': 'c52bdc376eee4bcd7a3fe80a8cb9baa76af7cb62eb3fc461df7f8a57f18c7114', '1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736': '383b6ed24c987030faf6c74ae37514db4b0164970a43be585e0e35657a165658', '1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097': 'd7c93df175bdcc13f38c7da7f3b56a02fc1cfe5e23370e0585ce4744ee3b670c', '1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018': 'b139834874c312fee9b277d0e6cf64c2e23fb343263edd48ca628c3162a32311', '1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941': '7b467b3dfa68d8b5962ccb2ed471865f15b550e9b713e9f29a08793ca4a7f047', '1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347': '8d3e4c3d79daee4279495d1b416842e617c2d90a5a8211676f59d14f81a3928f', '1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365': 'd2c28de35386f4a3bb9679f47df00d1507115704bb16b29a0deb21ac95d5d8b1', '1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449': '1e998b960f5e7a09e10d7bcf25296e64f654bbb42080b4d8b3b9a785d54701bf', '1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097': '07be0010d553cb02894e3e9f1806370b2ca733e49f6a80a13ace36fbda4d08bb', '1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678': 'ef608e1ce4821520037b563556268da90b3ac7528935e813d2f9978c0c9ebce7', '1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149': '632aa73374f3fd8887cb733b7bec2a10df96cd0e9f10baf1c4ca32084efa4e9c', '1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863': 'd531a17770406af0612d61a0fb297fb3435cef6d230aa6f13224ad2f8253610a', '1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557': '3212044e3cd8448c0b5147a3c8e45c56f740f0b949a806242a2579fd7a1a3be4', '1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364': '5cfc0466d5327e6bde607c8db3f2439af96070a0ba4bdd6f26ec0c4a5bff2956', '1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659': 'd7f1c4847cb3fea5a184746efd2511faf233ea11b33ad26c165a6701eb3892da', '1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636': '2f8f0634d6a1bc8c158adeff6631e39f897b3c6c201030a8733d3ab162921b2c', '1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100': '97da9d507ad66c91bac362c5057c09df33f84aed0f339f93b51c525e3c98a273', '1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882': '8725cb0420a2bd31ddf0778ef01ed9f126836194d8ac725ddb7911b8184c6e7b', '1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339': '03761a54fef47f3a8f598915a28282f852a387ca48ed6062747b4f266c24b574', '1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343': 'ea206fe68379c814ded9f2127ffe59b614c037627726ede6d4be8a7536b4b006', '1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834': 'b67e9485ebdda237931696bf624c427181ce28b1f411df45e5a7d4cddd1bcbbf', '1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833': 'ded4fd6b734442a5a7400f1b4b21ef80fc34d3d275df78cbfb1453a9900fb88b', '1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586': '2f8d2ba641034ce3c0fe1d02f61b42d203dea20046e020a7540a15691f201bf9', '1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166': 'ec8db386ab22463c352c33a9d56af4b524524e03d4d9de7c53d69bbbaab5a8a8', '1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864': 'f2820af0f8b0a1743c04b9473cd3b2b2ce7ac46015b3b70e6cc256084c84fa32', '1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471': '68710495a805a95891e69d3e2332ba4896d33acbe4dbb2fba7bbf48bc53a8297', '1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104': 'aa8a4ed67d98df66057968d37555c28c3657fe18666b36e02b179c0e209f1045', '1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182': 'a1bb4455e2b7fae3483ed006400255612083867d7a916c2dd4e16694313f833f', '1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314': '66e254757473db17841415dc04b6d35ca74d42c75cab687fcd09636db2f21b5f', '1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824': 'a31e871f7a14bba726eeabba64304434a5bfad73a7b77bc81e345e88959e8631', '1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586': '7f9601fde1c4644a2b40074d7dc7766cf71f765b67b7d49a12afb41b4b4868ec', '1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725': '2494005c887243a6d9f0a46d0ad446e6ae415a73879bdd724ff313c762415c8b', '1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221': '58930805c4f7e71753b5d09b3accefb1b47e29417ee22b2c25e1bcdd641f4b08', '1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019': 'eb7a7e1b152211f74e193f02d81c5ff7a23b3d0a6e0bef49cfd4fd045111cc1a', '1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790': '73671e3031f00c99aa48bee0d4f7befde90e226984009d1ffbda002cadd65a64', '1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205': '36b2f9ffcfc32998dff42d22f6eddb311f4d0c7d8586a8bde48afc05423b2adb', '1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320': 'b274f18d7a68b08ee744ff119cb761ea73007390c467c7104f854d8ab57e2912', '1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930': '01946b6a30dac0e66bc91d27595aaf9293786d153df0755ce7c87d247465d8e7', '1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484': 'fedf93c3d3613cfb7b4af6a1d697b7afc7990d92c1927af4286f9103be550eaf', '1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536': 'f66ae719828c52c6195105891932d5ac3f2440421c414940bcc581f7f6450e9e', '1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161': '2e49547cc5503e7993d78bda85c687409b20db5746f2d920b840ce947b31aa22', '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269': '312698f9b2cded32321570f4c0c614026dbe63a5cd4a1a7285bc84ddf23e966c', '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052': '17d93ba34e6542a7d5f3703318273893d582a86c52b38f36dbb3ba2ea3a938a2', '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580': 'fc43c45032ca80b1e349428e65d9e0a4282383089af300098e6a50d66d548e7b', '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238': '5740254c69e2df2cc72354533388a7e79bf2e3ef37a5bcc65581bd53ab2d8842', '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935': '316659408a8eec68efa23a5ee7e1622f8e9d22e879fc3c752909b6ffb12e2fd4', '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840': 'bf4393001352bff64ab0406dbf94f1f9d641a37e01661bd1582ee0c525bda712', '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756': '448c6f72d674e93d7e958a2b9d3233b2fde97fbe045c781ebc35fae273d7f5a6', '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604': '9a5aeebc561e8ffa95cbf650d4e63153cdbb71420dc2922e96fa8fe2db3a23fe', '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581': '0456b67b3138faf8ccfdd6dbefa9d09b839fbc64cd17a23d3aa4e4047d1a2431', '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945': '3b36a4c4456e69adf13b2980cba58e4fed2aafcc5b1497b07e9baa14a563fc9b', '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819': '88ea5d202e4c0e2a9a6bda73b0049124519757b41ee211092711dd0e11660dc1', '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973': '5976e46c110ab903caf7229943525bcbbae6ed345019b71cc92dbbd20661d3ea', '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492': '7f316b55af4018c7e42760a18a3640062ddcb87d7558206e300ca8ed70a694a6', '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803': 'c0cedb5809ec9b57e1560db0376814b2b267d0733bf6058e32c035d1352570ae', '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891': '7282696bb318b13647e4fb0626e63b9be7d776e4b7d6e249bb38c48d5926b217', '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903': 'be8b3a5e518ab3a02b8e123e36e58a41fd9dc08d3843372fa33bdc340db7282d', '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309': '11fd8841bf8fc28320bc557e797ea1466fc15b8f1e1bab72941ea9c00519c468', '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192': '62779941cba4075aa44ffb4bff5ec54e3c06e65b48a594c1174c366b4eeb13b8', '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711': '08e011d8b3527105e277ddaade3d54b4f85121db79afaf50362eaebc46282bcb', '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378': '600b9a669ff6d10b0eb5b183469695c1fbe6f88ece795ebe1fdc667aa2094630', '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404': '9c7a95f24207fd5202b3d1eb0be3ef8a7a2d4f18c93ae4c5d648613e1ad7846e', '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645': '1441191cf8492d8493a1715d8f15c54e6f0f0719fcd9dca25177aaa12c3db7d7', '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465': '062ccd95d280022c28960c0da542aca68c9edeccf2ec53944f3698aa65dbc39f', '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924': 'a9bf9afc548a3c78e7a5b846416033a8c1b7e19cc226e945375b5fafe313d817', '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608': 'c50b1899a15ca67bf684b897be19f152ea6b40bd1412dff1d88677f88f2b3323', '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550': 'ce4e2f70e5c64eaf40e81e74153383ad924b00d3567d8f662c4b7b0bab59086c', '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911': 'cdc2c598535c09dbcc343fed0630c8492b8393bbb31b5c32ff331c8b6f87f2d5', '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945': '814c0d1c8fa8b4040a07529293d122718315bab5560415fc52a5ebe04da37696', '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742': '09624bd36bd1dfd7f904f0778009bdf4b251966d69135d43fccee4f4809d2e11', '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683': 'e49c001b75c6421e3a167e041096c3f58566f83d9e40d84bfae172e2572e0e08', '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500': 'f93f9513da65d630f00977c01a37b30f652d5b0e12d54c57a7525dd44d50d60c', '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125': 'f4099ce555e080bae1681cc2254a04a299c4cf93946fc4f371dfe4c71007ec0b', '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187': '6e5ad843ba0dd40cdc12fe2aac04a254eeeb809aa0f4b2db3864766425271a2c', '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274': '87d8507fe376dbe2b24381584a8f3b614a1ed132fdb52577b9e0842c8e94ba52', '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132': '68b0f4365cf3e68395455c3f2cdb482bfce8b07a761f968dcdc4d4f764537153', '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450': 'eb235f47d184a951bbcb552b256856c8fb76b57fecf90940709ba45e2ab7998d', '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363': 'f38730b4342325bcbcec6d21c9da7dd8a2f9ca9afb273e8e0e2f85aa871e4a6a', '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710': 'cf26d4f1adab2314397169418b97797e8f348707b67c9ffbc7f1fda17e589905', '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395': '95bc085b8592d4dc437661a5095e50f55972101b2bd7311138871f931c8a650c', '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236': 'b275954ba941ed1de6d3308b56296f1a6403ace1d445ab2e69342ad64ba3019d', '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910': 'a3e9f3fb9a64f1194819c45cbb1c6807e65756ec4661b5bf43dc7d24ce7b20de', '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039': 'eca654fefd26b8d6a6dbd3ded0b438163dcba0f1238b1bc7f1f4f685141a714e', '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327': '64c23bf896bc7b10b7c57ca905549160209800e5297ff3f69b26de2b63152bba', '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774': 'c4ac94231997eac51c587fa50e653e394364cbc2762b36c167d13974b7ba69ad', '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002': '24b6b6464550b20898160795cbc89e1f0b2df03395dbbee727d72251cb5969d4', '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921': '387f2ff6528112c9fe7682c73eef15ffb2218458f6e0c9c2d0c104e340c760e7', '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220': 'e5657534232c0062779a55081b4656a97ae57a0afce737049488d78b9e801175', '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995': '0af704bbe5efe72dda27df7647340ff9e7d81f7ab4987310cf93f859101f1a8a', '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190': '73ba04177eae689e44260ca8857716c6285adef1a9e90563e24db150703dfe10', '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003': '4989627ee9ca00066131a0c9bca53743d3535678a23bea6500457734b57da101', '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828': '477f863a075bb323c4d68c77b5afe764cf06b1b3f99092dd75af42e26ef9b541', '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254': '20bbfead0f4e513d80ad36f680267f36d9358f9e40a5fcf7cc0051d7add60b56', '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037': '141946ea1ad73d6b83fdad8529760e333f2ed277e977ac48f6783ab54013553e', '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435': '303fcb9ea5dd3c4fcb2d1c8ebe1c735ac57346d51d818a5dfb96c0c0314437c5', '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767': 'e2411b89e5a4fc547215975ac09ad56e43711c2967b66fe1cfb0343adc1f0b2b', '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312': 'de1e22ca9d9f739ff20ae3ee53c5c43b219b02ba8d73de5dc76b4f824b7ee0f9', '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662': 'abd6bee29e08fcfeb1b822d4c61cb614359ab2f9d16aa315742da6877adfe423'}
    benchmark_inputs_present = all((data/'train_series'/uid).is_dir() for uid in benchmark_uids)
    if not benchmark_inputs_present:
        print('Benchmark skipped: training inputs are not fully mounted')
    if benchmark_inputs_present:
        benchmark_root = Path('/tmp/rsna-inference-benchmark')
        benchmark_root.mkdir(parents=True, exist_ok=True)
        with (benchmark_root / 'studies.csv').open('w', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=['StudyInstanceUID'])
            writer.writeheader()
            writer.writerows([{'StudyInstanceUID': uid} for uid in benchmark_uids])
        benchmark = run_offline_inference(
            checkpoint=Path(asset['checkpoint']), series_csv=data/'train_series.csv',
            dicom_root=data/'train_series', study_csv=benchmark_root/'studies.csv',
            dest=Path('/kaggle/working/benchmark_predictions.csv'), work=benchmark_root/'cache',
        )
        actual = {uid: sha256_file(benchmark_root/'cache'/f'{uid}.npy') for uid in benchmark_uids}
        assert actual == expected_shard_hashes, 'Train/inference pixel cache parity failed'
        assert not benchmark['quarantine']
        assert benchmark['runtime']['n_measured'] >= 30
        assert benchmark['runtime']['projected_seconds'] < 9*3600
        proof = {
            'benchmark': benchmark, 'parity_shard_hashes': actual, 'preprocess_hash': json.loads((benchmark_root/'cache/manifest.json').read_text())['preprocess_hash'],
            'checkpoint_sha256': sha256_file(Path(asset['checkpoint'])),
            'source_hashes': {str(path.relative_to(asset['src'])): sha256_file(path) for path in sorted(Path(asset['src']).rglob('*.py'))},
            'wheel_hashes': {path.name: sha256_file(path) for path in sorted((asset['root']/'wheels').glob('*.whl'))},
            'runtime_packages': {name: importlib.metadata.version(name) for name in ['numpy','torch','pillow','pydicom','pydantic','pyyaml','scikit-learn','filelock','pylibjpeg','pylibjpeg-libjpeg','pylibjpeg-openjpeg','python-gdcm']},
            'max_rss_kib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            'test_submission_sha256': sha256_file(Path('/kaggle/working/submission.csv')),
            'test_metadata_sha256': sha256_file(data/'test.csv'),
            'test_series_sha256': sha256_file(data/'test_series.csv'),
            'internet_required': False, 'lmstudio_required': False, 'drive_required': False,
        }
        Path('/kaggle/working/offline_proof.json').write_text(json.dumps(proof, indent=2))
        print(json.dumps(proof, indent=2))
